# Scrape BPS — Grup A (Statistics-Table BPS)

Grup A di skema = 12 card yang sumbernya semua dari **BPS** (tabel dinamis / statistics-table di bps.go.id) — artinya semuanya bisa ditarik lewat **Web API BPS** yang sama seperti notebook referensi kamu (`var` → `th` → `data`).

Karena ada 12 card tapi cuma **9 tabel unik** (beberapa card share 1 tabel — misal "Pertumbuhan Ekonomi" & "Kontribusi PDB" sama-sama dari tabel PDB Triwulanan), notebook ini generalize pattern referensi jadi loop multi-indikator, bukan ditulis manual satu-satu per card.

Alur:
1. Bangun katalog `var` BPS sekali aja (cache ke CSV, biar ga hit API berkali-kali tiap re-run)
2. Cari `var_id` tiap indikator otomatis (keyword search + cocokin ke judul resmi tabel dari skema kamu) — **tetap perlu di-review manual**, auto-match bisa salah kalau keyword-nya umum
3. Ambil tahun tersedia per indikator
4. Ambil semua tahun data, gabung jadi 1 dataframe per indikator (time series lengkap, bukan cuma tahun terakhir)
5. Simpan CSV ke `data/raw/groupA/`, 1 file per card dengan nama `{no}_{nama_tabel_db}.csv`

In [2]:
import os, time, requests, pandas as pd
from difflib import SequenceMatcher
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("BPS_API_KEY")
assert API_KEY, "API key tidak ditemukan! Cek file .env kamu."

BASE = "https://webapi.bps.go.id/v1/api/list"
SESSION = requests.Session()

def bps_get(params, retries=3, backoff=2):
    """Sama seperti reference notebook, ditambah retry ringan buat transient error."""
    for attempt in range(1, retries + 1):
        try:
            resp = SESSION.get(BASE, params={**params, "key": API_KEY}, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            assert data["status"] == "OK", f"API error: {data}"
            return data
        except (requests.RequestException, AssertionError):
            if attempt == retries:
                raise
            time.sleep(backoff * attempt)

## 1. Bangun katalog variabel BPS (cache sekali, reuse berkali-kali)

Referensi kamu loop semua page `model=var` tiap kali nyari 1 keyword → kalau nyari 9 keyword, itu refetch full katalog 9x. Di sini ditarik **sekali**, disimpan ke CSV lokal, dan dipakai ulang buat semua indikator (load dari cache kalau file udah ada).

In [3]:
CATALOG_PATH = "bps_var_catalog.csv"

def build_var_catalog(force_refresh=False):
    if os.path.exists(CATALOG_PATH) and not force_refresh:
        return pd.read_csv(CATALOG_PATH, dtype=str)

    resp = bps_get({"model": "var", "lang": "ind", "domain": "0000", "page": 1})
    pages = resp["data"][0]["pages"]

    semua_var = []
    for page in range(1, pages + 1):
        r = bps_get({"model": "var", "lang": "ind", "domain": "0000", "page": page})
        semua_var.extend(r["data"][1])
        time.sleep(0.3)

    df_var = pd.DataFrame(semua_var)
    df_var.to_csv(CATALOG_PATH, index=False)
    return df_var

df_var = build_var_catalog()
print(f"Total variabel di katalog: {len(df_var)}")
df_var.head()

Total variabel di katalog: 1701


,var_id,title,sub_id,sub_name,subcsa_id,subcsa_name,def,notes,vertical,unit,graph_id,graph_name
0,70,Persentase Penduduk Usia 5 tahun ke atas yang ...,2,Komunikasi,565.0,Masyarakat Informasi,NaN,&lt;p&gt;&lt;br /&gt;&lt;/p&gt;&lt;p&gt;Sumber...,1,Tidak Ada Satuan,1.0,bar
1,111,Persentase Rumah Tangga yang Memiliki/Menguasa...,2,Komunikasi,565.0,Masyarakat Informasi,NaN,&lt;p&gt;&lt;br /&gt;&lt;/p&gt;&lt;p&gt;Sumber...,1,Tidak Ada Satuan,1.0,bar
2,391,Persentase Rumah Tangga yang Memiliki/Menguasa...,2,Komunikasi,565.0,Masyarakat Informasi,NaN,&lt;p&gt;&lt;br /&gt;&lt;/p&gt;&lt;p&gt;Sumber...,1,Tidak Ada Satuan,1.0,bar
3,392,Rata-Rata Banyaknya Nomor Telepon Seluler Akti...,2,Komunikasi,565.0,Masyarakat Informasi,NaN,"&lt;p&gt;Sumber: BPS, Survei Sosial Ekonomi Na...",1,Tidak Ada Satuan,1.0,bar
4,393,Persentase Rumah Tangga yang Memiliki/Menguasa...,2,Komunikasi,565.0,Masyarakat Informasi,NaN,"&lt;p&gt;Sumber: BPS, Survei Sosial Ekonomi Na...",1,Tidak Ada Satuan,1.0,bar


## 2. Daftar indikator Grup A

9 tabel BPS unik, dipetakan ke 12 card sesuai sheet `Data Lengkap` di file skema kamu. `judul_target` diambil dari kolom **Judul Resmi Tabel** — dipakai buat auto-matching, bukan cuma keyword doang (karena 1 keyword sering nyangkut ke banyak `var_id`).

Tiap `card` dilengkapi `no` (kolom **No** di sheet) dan `tabel_db` (kolom **Tabel DB**) — dua ini dipakai buat nentuin nama file output (lihat section 6). Beberapa indikator dipakai 2 card (1 tabel BPS → 2 nama tabel DB berbeda), jadi nanti hasilnya disimpan jadi 2 file terpisah dengan data yang sama.

In [4]:
INDIKATOR_GRUP_A = [
    {
        "slug": "pdb_triwulanan",
        "keyword": "pdb triwulanan",
        "judul_target": "PDB Triwulanan Atas Dasar Harga Konstan menurut Pengeluaran (Milyar Rupiah)",
        "cards": [
            {"no": 1, "nama": "Pertumbuhan Ekonomi", "tabel_db": "usecase_ekonomi.ekonomi_pertumbuhan_ekonomi_kuartal"},
            {"no": 17, "nama": "Kontribusi PDB", "tabel_db": "usecase_ekonomi.ekonomi_kontribusi_investasi_pdb_kuartal"},
        ],
    },
    {
        "slug": "inflasi_provinsi",
        "keyword": "inflasi tahunan",
        "judul_target": "Inflasi Tahunan (Y-on-Y) 38 Provinsi (2022=100) (Persen)",
        "cards": [
            {"no": 12, "nama": "Inflasi Daerah", "tabel_db": "usecase_ekonomi.ekonomi_inflasi_provinsi_ranking"},
        ],
    },
    {
        "slug": "angkatan_kerja_provinsi",
        "keyword": "angkatan kerja",
        "judul_target": "Angkatan Kerja (AK) menurut Provinsi (Orang)",
        "cards": [
            {"no": 30, "nama": "Jumlah Angkatan Kerja", "tabel_db": "usecase_ekonomi.ekonomi_jumlah_angkatan_kerja"},
        ],
    },
    {
        "slug": "penduduk_bekerja_pengangguran",
        "keyword": "penduduk bekerja",
        "judul_target": "Jumlah dan Persentase Penduduk Bekerja dan Pengangguran",
        "cards": [
            {"no": 32, "nama": "Penduduk Bekerja", "tabel_db": "usecase_ekonomi.ekonomi_penduduk_bekerja"},
        ],
    },
    {
        "slug": "tpak_provinsi",
        "keyword": "tingkat partisipasi angkatan kerja",
        "judul_target": "Persentase Angkatan Kerja Terhadap Penduduk Usia Kerja (TPAK) menurut Provinsi (Persen)",
        "cards": [
            {"no": 33, "nama": "Tingkat Partisipasi", "tabel_db": "usecase_ekonomi.ekonomi_tingkat_partisipasi_angkatan_kerja"},
        ],
    },
    {
        "slug": "tpt_provinsi",
        "keyword": "tingkat pengangguran terbuka",
        "judul_target": "Tingkat Pengangguran Terbuka Menurut Provinsi (Persen)",
        "cards": [
            {"no": 34, "nama": "Tingkat Pengangguran", "tabel_db": "usecase_ekonomi.ekonomi_tingkat_pengangguran"},
            {"no": 36, "nama": "TPT By Province", "tabel_db": "usecase_ekonomi.ekonomi_tpt_by_province"},
        ],
    },
    {
        "slug": "wisman_bulanan_kebangsaan",
        "keyword": "wisatawan mancanegara",
        "judul_target": "Jumlah Kunjungan Wisatawan Mancanegara per bulan Menurut Paspor yang Dipegang (Kunjungan)",
        "cards": [
            {"no": 43, "nama": "Wisatawan Mancanegara", "tabel_db": "usecase_ekonomi.ekonomi_wisatawan_mancanegara"},
            {"no": 44, "nama": "Wisatawan Mancanegara Asal", "tabel_db": "usecase_ekonomi.ekonomi_wisatawan_mancanegara_asal_negara"},
        ],
    },
    {
        "slug": "wisman_pintu_masuk",
        "keyword": "wisatawan mancanegara",
        "judul_target": "Jumlah Kedatangan Wisatawan Mancanegara ke Indonesia Menurut Pintu Masuk (Orang)",
        "cards": [
            {"no": 46, "nama": "Wisatawan Mancanegara Entry Points", "tabel_db": "usecase_ekonomi.ekonomi_wisatawan_mancanegara_entry_points"},
        ],
    },
    {
        "slug": "wisnus_provinsi_tujuan",
        "keyword": "wisatawan nusantara",
        "judul_target": "Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan (Perjalanan)",
        "cards": [
            {"no": 47, "nama": "Wisatawan Nusantara", "tabel_db": "usecase_ekonomi.ekonomi_wisatawan_nusantara"},
        ],
    },
]

len(INDIKATOR_GRUP_A)

9

## 3. Auto-match `var_id` (keyword filter + similarity ke judul resmi)

Untuk tiap indikator: filter katalog by keyword (case-insensitive substring di title), terus ranking kandidat berdasarkan kemiripan title ke `judul_target` (pakai `difflib.SequenceMatcher`). Hasilnya ditaruh di `df_match` buat kamu cek manual.

In [5]:
def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def find_best_var(df_var, keyword, judul_target, top_n=5):
    mask = df_var["title"].str.lower().str.contains(keyword.lower(), na=False)
    candidates = df_var[mask].copy()
    if candidates.empty:
        return candidates, None
    candidates["score"] = candidates["title"].apply(lambda t: similarity(t, judul_target))
    candidates = candidates.sort_values("score", ascending=False)
    return candidates.head(top_n), candidates.iloc[0]

match_rows = []
auto_match = {}
for ind in INDIKATOR_GRUP_A:
    candidates, best = find_best_var(df_var, ind["keyword"], ind["judul_target"])
    if best is None:
        match_rows.append({"slug": ind["slug"], "var_id": None, "score": 0, "title_ditemukan": "(TIDAK KETEMU, cek keyword)"})
        continue
    auto_match[ind["slug"]] = best["var_id"]
    match_rows.append({
        "slug": ind["slug"],
        "var_id": best["var_id"],
        "score": round(best["score"], 2),
        "title_ditemukan": best["title"],
    })

df_match = pd.DataFrame(match_rows)
df_match

,slug,var_id,score,title_ditemukan
0,pdb_triwulanan,1634,0.79,[Seri 2000] 2. PDB Triwulanan Atas Dasar Harga...
1,inflasi_provinsi,2263,0.91,Inflasi Tahunan (Y-on-Y) 38 Provinsi (2022=100)
2,angkatan_kerja_provinsi,2394,0.90,Angkatan Kerja (AK) menurut Provinsi
3,penduduk_bekerja_pengangguran,1953,1.00,Jumlah dan Persentase Penduduk Bekerja dan Pen...
4,tpak_provinsi,2200,0.41,Tingkat Partisipasi Angkatan Kerja Menurut Jen...
5,tpt_provinsi,543,0.91,Tingkat Pengangguran Terbuka Menurut Provinsi
6,wisman_bulanan_kebangsaan,1470,0.93,Jumlah Kunjungan Wisatawan Mancanegara per bul...
7,wisman_pintu_masuk,1017,0.95,Jumlah Kedatangan Wisatawan Mancanegara ke Ind...
8,wisnus_provinsi_tujuan,2201,0.90,Jumlah Perjalanan Wisatawan Nusantara Menurut ...


### Review & override manual

Cek `df_match` di atas. Kalau ada `score` rendah (kira-kira di bawah ~0.6) atau title-nya keliatan salah, lihat kandidat lain dulu:

```python
find_best_var(df_var, "keyword kamu", "judul target")[0]
```

lalu isi `var_id` yang benar di `MANUAL_OVERRIDE` di bawah.

In [6]:
MANUAL_OVERRIDE = {
    # "slug_indikator": "var_id_yang_benar",
}

FINAL_VAR_ID = {**auto_match, **MANUAL_OVERRIDE}
FINAL_VAR_ID

{'pdb_triwulanan': '1634',
 'inflasi_provinsi': '2263',
 'angkatan_kerja_provinsi': '2394',
 'penduduk_bekerja_pengangguran': '1953',
 'tpak_provinsi': '2200',
 'tpt_provinsi': '543',
 'wisman_bulanan_kebangsaan': '1470',
 'wisman_pintu_masuk': '1017',
 'wisnus_provinsi_tujuan': '2201'}

## 4. Cek tahun tersedia per indikator

In [7]:
def get_available_years(var_id):
    resp_th = bps_get({"model": "th", "lang": "ind", "domain": "0000", "var": var_id})
    return pd.DataFrame(resp_th["data"][1])  # kolom: th_id, th

th_per_indikator = {}
for slug, var_id in FINAL_VAR_ID.items():
    if var_id is None:
        continue
    df_th = get_available_years(var_id)
    if df_th.empty:
        print(f"{slug:30s} var_id={var_id} -> tidak ada tahun ditemukan, skip")
        continue
    th_per_indikator[slug] = df_th
    print(f"{slug:30s} var_id={str(var_id):>6}  tahun: {df_th['th'].min()}-{df_th['th'].max()} ({len(df_th)} titik)")
    time.sleep(0.3)

pdb_triwulanan                 var_id=  1634  tahun: 2005-2014 (10 titik)
inflasi_provinsi               var_id=  2263  tahun: 2024-2026 (3 titik)
angkatan_kerja_provinsi        var_id=  2394  tahun: 2025-2025 (1 titik)
penduduk_bekerja_pengangguran  var_id=  1953  tahun: 2017-2026 (10 titik)
tpak_provinsi                  var_id=  2200  tahun: 2018-2025 (8 titik)
tpt_provinsi                   var_id=   543  tahun: 2017-2026 (10 titik)
wisman_bulanan_kebangsaan      var_id=  1470  tahun: 2017-2026 (10 titik)
wisman_pintu_masuk             var_id=  1017  tahun: 2006-2025 (10 titik)
wisnus_provinsi_tujuan         var_id=  2201  tahun: 2019-2026 (8 titik)


## 5. Ambil semua tahun data per indikator

Parsing-nya sama kayak referensi (`vervar` x `turvar` x `tahun` x `turtahun`), tapi di-loop per `th_id` biar dapet seluruh time series, bukan cuma tahun terakhir. Kolom dimensi utama dinormalisasi jadi `vervar` (isinya bisa Provinsi, Jenis Kendaraan, dll — namanya disimpan di kolom `label_vervar`) biar gampang digabung antar indikator.

Tiap indikator ditarik di `try/except` sendiri biar 1 error ga nge-stop semua proses.

In [8]:
def fetch_one_year(var_id, th_id):
    resp_data = bps_get({"model": "data", "lang": "ind", "domain": "0000", "var": var_id, "th": th_id})

    map_vervar   = {str(v["val"]): v["label"] for v in resp_data["vervar"]}
    map_turvar   = {str(v["val"]): v["label"] for v in resp_data["turvar"]}
    map_tahun    = {str(v["val"]): str(v["label"]) for v in resp_data["tahun"]}
    map_turtahun = {str(v["val"]): v["label"] for v in resp_data["turtahun"]}
    var_id_str   = str(resp_data["var"][0]["val"])
    label_vervar = resp_data["labelvervar"]

    rows = []
    for ver_id, ver_label in map_vervar.items():
        for tur_id, tur_label in map_turvar.items():
            for th_key, th_label in map_tahun.items():
                for turth_id in map_turtahun.keys():
                    key   = f"{ver_id}{var_id_str}{tur_id}{th_key}{turth_id}"
                    nilai = resp_data["datacontent"].get(key)
                    if nilai is not None:
                        rows.append({
                            "vervar": ver_label,
                            "jenis" : tur_label,
                            "tahun" : th_label,
                            "nilai" : nilai,
                        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df.insert(1, "label_vervar", label_vervar)
    return df

def fetch_indikator_data(var_id, df_th, sleep=0.4):
    frames = []
    for _, row in df_th.iterrows():
        df_year = fetch_one_year(var_id, row["th_id"])
        frames.append(df_year)
        time.sleep(sleep)
    df_full = pd.concat(frames, ignore_index=True)
    df_full.insert(0, "var_id", var_id)
    return df_full

hasil_per_indikator = {}
gagal = []
for slug, var_id in FINAL_VAR_ID.items():
    if var_id is None or slug not in th_per_indikator:
        gagal.append((slug, "var_id/tahun tidak ditemukan"))
        continue
    try:
        df_hasil = fetch_indikator_data(var_id, th_per_indikator[slug])
        df_hasil.insert(0, "indikator", slug)
        hasil_per_indikator[slug] = df_hasil
        print(f"OK    {slug:30s} {df_hasil.shape[0]} baris")
    except Exception as e:
        gagal.append((slug, str(e)))
        print(f"GAGAL {slug:30s} {e}")

if gagal:
    print("\nIndikator gagal / butuh dicek manual:")
    for slug, alasan in gagal:
        print(f" - {slug}: {alasan}")

OK    pdb_triwulanan                 1200 baris
OK    inflasi_provinsi               1131 baris
OK    angkatan_kerja_provinsi        312 baris
OK    penduduk_bekerja_pengangguran  76 baris
OK    tpak_provinsi                  8798 baris
OK    tpt_provinsi                   685 baris
OK    wisman_bulanan_kebangsaan      30008 baris
GAGAL wisman_pintu_masuk             'vervar'
OK    wisnus_provinsi_tujuan         3549 baris

Indikator gagal / butuh dicek manual:
 - wisman_pintu_masuk: 'vervar'


## 6. Simpan ke CSV (per card, nama file = `{no}_{nama_tabel_db}`)

Output disimpan ke `data/raw/groupA/`. Tiap **card** (bukan tiap indikator BPS) jadi 1 file, dinamain dari kolom **No** + nama tabel DB-nya (tanpa prefix `usecase_ekonomi.`) — misal row 1 `usecase_ekonomi.ekonomi_pertumbuhan_ekonomi_kuartal` jadi `1_ekonomi_pertumbuhan_ekonomi_kuartal.csv`. Kalau 1 tabel BPS dipakai 2 card, datanya sama tapi disimpan 2x dengan nama beda — biar konsisten sama struktur tabel DB kamu.

In [ ]:
OUTPUT_DIR = "../data/raw/groupA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CARDS_MAP = {ind["slug"]: ind["cards"] for ind in INDIKATOR_GRUP_A}

def tabel_db_to_filename(no, tabel_db):
    # "usecase_ekonomi.ekonomi_pertumbuhan_ekonomi_kuartal" -> "1_ekonomi_pertumbuhan_ekonomi_kuartal.csv"
    nama_tabel = tabel_db.split(".", 1)[-1]
    return f"{no}_{nama_tabel}.csv"

summary = []
all_frames = []
for slug, df_hasil in hasil_per_indikator.items():
    for card in CARDS_MAP.get(slug, []):
        df_card = df_hasil.copy()
        df_card["card"] = card["nama"]
        df_card["tabel_db"] = card["tabel_db"]
        fname = tabel_db_to_filename(card["no"], card["tabel_db"])
        fpath = os.path.join(OUTPUT_DIR, fname)
        df_card.to_csv(fpath, index=False)
        all_frames.append(df_card)
        summary.append({"no": card["no"], "indikator": slug, "card": card["nama"], "file": fname, "baris": df_card.shape[0]})

if all_frames:
    df_gabungan = pd.concat(all_frames, ignore_index=True)
    gabungan_path = os.path.join(OUTPUT_DIR, "0_gabungan_grupA.csv")
    df_gabungan.to_csv(gabungan_path, index=False)
    print(f"Tersimpan -> {gabungan_path} | {df_gabungan.shape[0]} baris, {df_gabungan.shape[1]} kolom")

pd.DataFrame(summary).sort_values("no")

Tersimpan -> data/raw/groupA\0_gabungan_grupA.csv | 77652 baris, 9 kolom


,no,indikator,card,file,baris
0,1,pdb_triwulanan,Pertumbuhan Ekonomi,1_ekonomi_pertumbuhan_ekonomi_kuartal.csv,1200
2,12,inflasi_provinsi,Inflasi Daerah,12_ekonomi_inflasi_provinsi_ranking.csv,1131
1,17,pdb_triwulanan,Kontribusi PDB,17_ekonomi_kontribusi_investasi_pdb_kuartal.csv,1200
3,30,angkatan_kerja_provinsi,Jumlah Angkatan Kerja,30_ekonomi_jumlah_angkatan_kerja.csv,312
4,32,penduduk_bekerja_pengangguran,Penduduk Bekerja,32_ekonomi_penduduk_bekerja.csv,76
5,33,tpak_provinsi,Tingkat Partisipasi,33_ekonomi_tingkat_partisipasi_angkatan_kerja.csv,8798
6,34,tpt_provinsi,Tingkat Pengangguran,34_ekonomi_tingkat_pengangguran.csv,685
7,36,tpt_provinsi,TPT By Province,36_ekonomi_tpt_by_province.csv,685
8,43,wisman_bulanan_kebangsaan,Wisatawan Mancanegara,43_ekonomi_wisatawan_mancanegara.csv,30008
9,44,wisman_bulanan_kebangsaan,Wisatawan Mancanegara Asal,44_ekonomi_wisatawan_mancanegara_asal_negara.csv,30008


## Catatan

- `pdb_triwulanan`, `tpt_provinsi`, dan `wisman_bulanan_kebangsaan` masing-masing dipakai 2 card sekaligus (lihat kolom `card` di hasil) — itu sesuai sheet `Data Lengkap`, bukan duplikat data yang perlu dihapus. Karena itu, jumlah file CSV di `data/raw/groupA/` = 12 (1 per card) + 1 gabungan, walaupun cuma 9 kali request unik ke API.
- Auto-match `var_id` di step 3 itu best-effort. Selalu cek `df_match` sebelum lanjut ambil data, terutama buat indikator dengan keyword umum (`inflasi`, `wisatawan mancanegara`) yang kandidatnya bisa banyak — `wisman_bulanan_kebangsaan` dan `wisman_pintu_masuk` sengaja pakai keyword yang mirip, jadi double-check dua-duanya kebagian `var_id` yang berbeda.
- Kalau ada indikator yang BPS-nya ternyata di-host bukan sebagai "tabel dinamis" tapi sebagai press release / dokumen PDF biasa (misal beberapa indikator ketenagakerjaan di Grup B yang "harus download berita resmi PDF dulu"), itu di luar scope Web API ini — perlu pipeline scraping/PDF-parsing terpisah.